In [0]:
from pyspark.sql.functions import col, sum, row_number
from pyspark.sql.window import Window

catalog = "workspace"
schema = "lakehouse_f1" 

df_silver = spark.read.table(f"{catalog}.{schema}.silver_lap_times")

In [0]:
janela_corrida = Window.partitionBy("raceId", "driverId").orderBy("lap")
df_gold = df_silver.withColumn("stint_id", sum("fez_pit_stop").over(janela_corrida))

In [0]:
janela_stint = Window.partitionBy("raceId", "driverId", "stint_id").orderBy("lap")
df_gold = df_gold.withColumn("idade_pneu_voltas", row_number().over(janela_stint))

In [0]:
nome_tabela_gold = f"{catalog}.{schema}.gold_pit_stop_features"
df_gold.write.format("delta").mode("overwrite").saveAsTable(nome_tabela_gold)

print("Camada Gold concluída! Tabela final de features criada com sucesso.")